# 01 — Collect SEC 10-K Filings

This notebook downloads 10-K filings for five companies (AAPL, MSFT, NVDA, TSLA, JPM) across fiscal years 2022–2024 directly from the SEC EDGAR REST API.

**What it does:**
1. Queries `https://data.sec.gov/submissions/CIK{cik}.json` for each company to find accession numbers
2. Downloads the primary filing document, strips HTML, saves as plain text
3. Writes `data/processed/filing_metadata.csv` with provenance for all downloads

**Expected output:** 15 `.txt` files in `data/raw/`, one per company-year combination.

**Runtime:** ~5–10 minutes (rate-limited to 0.15s between requests per SEC guidelines).

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
from pathlib import Path

from src.config import COMPANIES, YEARS, RAW_DIR, PROCESSED_DIR
from src.sec_loader import load_all_filings, get_filing_metadata

print("Companies:", list(COMPANIES.keys()))
print("Years:", YEARS)
print(f"Expected filings: {len(COMPANIES) * len(YEARS)}")

## 1.1 — Download all filings

This cell runs the full download pipeline. It will skip files that are already present on disk (idempotent). Progress is logged to stdout.

In [ ]:
metadata_df = load_all_filings(save_dir=RAW_DIR)
metadata_df

## 1.2 — Inspect the metadata table

Check that we have a row for each company × year, verify accession numbers, and identify any download failures.

In [ ]:
print(f"Total filings attempted: {len(metadata_df)}")
print(f"Successful downloads: {metadata_df['local_path'].notna().sum()}")
print(f"Failures: {metadata_df['local_path'].isna().sum()}")

# Show the pivot table: company × year
pivot = metadata_df.pivot_table(
    index='ticker',
    columns='year',
    values='filing_date',
    aggfunc='first'
)
print("\nFiling dates by company × year:")
pivot

In [ ]:
# Check for any rows that failed
failures = metadata_df[metadata_df['local_path'].isna()]
if len(failures) > 0:
    print("Failed downloads:")
    print(failures[['ticker', 'year', 'error']])
else:
    print("All filings downloaded successfully.")

## 1.3 — Spot-check one filing

Load NVDA 2024 and print the first 3000 characters to verify the HTML was stripped and the text is readable.

In [ ]:
spot_check = metadata_df[(metadata_df['ticker'] == 'NVDA') & (metadata_df['year'] == 2024)]
if len(spot_check) > 0 and spot_check.iloc[0]['local_path']:
    path = spot_check.iloc[0]['local_path']
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"File: {path}")
    print(f"Total characters: {len(text):,}")
    print(f"Total lines: {text.count(chr(10)):,}")
    print("\n--- First 3000 characters ---")
    print(text[:3000])
else:
    print("NVDA 2024 not available for spot check.")

## 1.4 — File size summary

Compare file sizes to verify we downloaded substantive content (10-K filings are typically 300KB–2MB of plain text).

In [ ]:
import os

size_records = []
for _, row in metadata_df[metadata_df['local_path'].notna()].iterrows():
    path = row['local_path']
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        size_records.append({
            'ticker': row['ticker'],
            'year': row['year'],
            'size_kb': round(size_kb, 1)
        })

size_df = pd.DataFrame(size_records)
print(size_df.pivot_table(index='ticker', columns='year', values='size_kb', aggfunc='first'))